In [1]:
import pandas as pd
import numpy as np

def evaluate_alpha_theory():
    # --- 1. Calculate dP/dT from NVT data ---
    print("--- Step 1: Extracting dP/dT ---")
    nvt_files = {
        0.000010: "berendsen_T0.000010_g0.dat",
        0.000015: "berendsen_T0.000015_g0.dat"
    }
    
    pressures = {}
    for T, filename in nvt_files.items():
        try:
            df = pd.read_csv(filename, sep=r'\s+', header=None, names=['Time', 'KE', 'PE', 'Pressure'])
            pressures[T] = df.iloc[len(df)//2:]['Pressure'].mean() # Discard warmup
        except FileNotFoundError:
            print(f"Error: {filename} not found.")
            return

    T1, T2 = 0.000010, 0.000015
    dp_dt = (pressures[T2] - pressures[T1]) / (T2 - T1)
    print(f"dP/dT = {dp_dt:.8f}")

    # --- 2. Extract Bulk Modulus (K) from Elasticity file ---
    print("\n--- Step 2: Extracting Bulk Modulus (K) ---")
    elasticity_file = "3dlj_elasticity_N4000_s000.dat"
    
    try:
        # Load the raw text data into a numpy array
        elastic_data = np.loadtxt(elasticity_file)
        print(f"Raw elasticity data: {elastic_data}")
        
        # IMPORTANT: You must point this index to the column that represents K.
        # Often, C-code elasticity outputs are formatted like: 
        # [Glass_Serial, Density, Pressure, Bulk_Modulus (K), Shear_Modulus (G)]
        # We assume K is at index 3 (the 4th value). Change this if necessary!
        if elastic_data.ndim == 1:
            K = elastic_data[3]  
        else:
            K = elastic_data[0, 4] # If multiple rows, take the first row
            
        print(f"Extracted K = {K:.8f}")
        
    except Exception as e:
        print(f"Error reading {elasticity_file}: {e}")
        return

    # --- 3. Final Alpha Calculation ---
    print("\n--- Step 3: Final Alpha Metric ---")
    alpha = dp_dt / K
    
    print(f"Alpha (thermal expansivity) = {alpha:.8f}")

# Run the complete pipeline
evaluate_alpha_theory()

--- Step 1: Extracting dP/dT ---
dP/dT = 5.30736957

--- Step 2: Extracting Bulk Modulus (K) ---
Raw elasticity data: [[ 0.00000000e+00 -5.84364001e+00 -5.25706325e-03  6.54571135e+00
   2.51394763e+01  5.54969138e-04  3.53949765e+01]
 [ 1.00000000e+00 -5.85446873e+00 -2.97244460e-02  5.88890856e+00
   2.49310385e+01  2.21942086e-01  3.41516629e+01]
 [ 2.00000000e+00 -5.85678995e+00 -5.39687386e-02  6.44974410e+00
   2.48755482e+01 -1.04352912e-01  2.96043127e+01]
 [ 3.00000000e+00 -5.84966048e+00 -3.05401561e-02  6.39476691e+00
   2.49816829e+01  2.23725784e-01  3.33694283e+01]
 [ 4.00000000e+00 -5.84284472e+00 -1.93717457e-02  6.29758277e+00
   2.51709005e+01  8.97129054e-02  4.14313502e+01]
 [ 5.00000000e+00 -5.84272598e+00 -5.40490143e-03  6.42531229e+00
   2.51305162e+01  1.87871545e-01  3.54671417e+01]
 [ 6.00000000e+00 -5.83924862e+00  1.02398813e-02  6.32128761e+00
   2.52391734e+01 -2.13364446e-01  3.09180708e+01]
 [ 7.00000000e+00 -5.84763393e+00 -8.40187974e-03  6.33859787e+